## Step 1 · Setup

**Student ID last 4 digits:** 2534
**Environment:** `.aselmuratbek_ML_task_1` (Python 3.12.6)
**Libraries:** numpy 2.0.2, pandas 2.3.3, matplotlib 3.9.4, scikit-learn 1.6.1

In [2]:
# Step 2 — Load and inspect
import pandas as pd

YOUR_ID = 2534

raw = pd.read_csv("data/almaty_apartments_raw.csv")
ids = raw["listing_id"].drop_duplicates().sample(frac=0.9, random_state=YOUR_ID)
df = raw[raw["listing_id"].isin(ids)].reset_index(drop=True)

print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nDescribe:")
print(df.describe(include="all").T)

Shape: (1159, 11)

Dtypes:
listing_id           object
listed_date          object
district             object
rooms                object
area_m2              object
floor                 int64
total_floors          int64
year_built          float64
building_type        object
ceiling_height_m    float64
price_kzt            object
dtype: object

Missing values:
listing_id            0
listed_date           0
district              0
rooms                 0
area_m2               0
floor                 0
total_floors          0
year_built           54
building_type         0
ceiling_height_m    139
price_kzt             0
dtype: int64

Describe:
                   count unique         top freq         mean         std  \
listing_id          1159   1080   ALM-10932    3          NaN         NaN   
listed_date         1159    248  2026-04-18   11          NaN         NaN   
district            1159     48       Medeu  151          NaN         NaN   
rooms               1159      6       

In [3]:
# Полный describe без обрезки
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 200)
print(df.describe(include="all").T)

                   count unique         top freq         mean         std  min     25%     50%     75%     max
listing_id          1159   1080   ALM-10932    3          NaN         NaN  NaN     NaN     NaN     NaN     NaN
listed_date         1159    248  2026-04-18   11          NaN         NaN  NaN     NaN     NaN     NaN     NaN
district            1159     48       Medeu  151          NaN         NaN  NaN     NaN     NaN     NaN     NaN
rooms               1159      6           2  442          NaN         NaN  NaN     NaN     NaN     NaN     NaN
area_m2             1159    654        42.0    7          NaN         NaN  NaN     NaN     NaN     NaN     NaN
floor             1159.0    NaN         NaN  NaN     5.899914    4.723475 -1.0     2.0     5.0     8.0    30.0
total_floors      1159.0    NaN         NaN  NaN    10.416739    6.045146  3.0     5.0     9.0    12.0    25.0
year_built        1105.0    NaN         NaN  NaN  1955.984615  259.608664  0.0  1972.0  1987.0  2009.0  2024.0
b

In [1]:
# Step 1 — Setup: versions
import sys, numpy, pandas, sklearn
print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.12.6 (v3.12.6:a4a2d2b0d85, Sep  6 2024, 16:08:03) [Clang 13.0.0 (clang-1300.0.29.30)]
NumPy: 2.0.2
pandas: 2.3.3
scikit-learn: 1.6.1


In [4]:
# Сырые значения — чтобы понять, почему pandas не распарсил
for col in ["area_m2", "price_kzt", "rooms", "listed_date"]:
    print(f"\n=== {col} ===")
    print(df[col].sample(8, random_state=1).tolist())


=== area_m2 ===
['35.0', '42.1', '25.9', '98.2', '55.0', '89.5', '148.4', '57.2']

=== price_kzt ===
['14350000', '37940000', '28630000', '93640000', '27310000', '87370000', '125650000', '40420000']

=== rooms ===
['1', '1', 'studio', '4', '2', '4', '5', '2']

=== listed_date ===
['2026-05-16', '2026-01-28', '2026-07-14', '2026-08-26', '2026-01-05', '2026-08-01', '2026-04-13', '2026-05-29']


In [5]:
# Grain — сколько дублей и репостов
print("Exact duplicates:", df.duplicated().sum())
print("Reposts (same listing_id, non-identical):",
      df.drop_duplicates()["listing_id"].duplicated().sum())

Exact duplicates: 33
Reposts (same listing_id, non-identical): 46


In [6]:
# Найти два примера репостов (один listing_id, разные строки)
dup_ids = df.drop_duplicates()["listing_id"].duplicated()
dup_list = df.drop_duplicates()[dup_ids]["listing_id"].head(2).tolist()
for lid in dup_list:
    print(f"\n=== listing_id={lid} ===")
    print(df[df["listing_id"] == lid][["listing_id","listed_date","price_kzt","area_m2","rooms"]])


=== listing_id=ALM-11147 ===
    listing_id listed_date price_kzt area_m2 rooms
152  ALM-11147  2026-06-25  57130000    56.7     2
163  ALM-11147  2026-05-26  59610000    56.7     2
355  ALM-11147  2026-06-25  57130000    56.7     2

=== listing_id=ALM-11017 ===
    listing_id listed_date price_kzt area_m2 rooms
59   ALM-11017  2026-04-30  13430000    30.3     1
215  ALM-11017  2026-06-25  12810000    30.3     1


In [7]:
print(df["district"].value_counts())

district
Medeu            151
Zhetysu          129
Bostandyk        128
Auezov           126
Turksib          125
Almaly           119
Nauryzbay        107
Alatau           105
medeu             11
turksib           11
Наурызбайский      9
auezov             7
Auezovskiy         7
bostandyk          7
Alatauskiy         6
AUEZOV             6
 Auezov            6
 Zhetysu           5
Медеуский          5
TURKSIB            5
 Turksib           5
Bostandykskiy      5
ALMALY             5
Almalinskiy        4
 Bostandyk         4
Nauryzbayskiy      4
ZHETYSU            4
 Almaly            4
 Medeu             4
 Alatau            4
nauryzbay          4
BOSTANDYK          4
MEDEU              4
ALATAU             3
Zhetysuskiy        3
NAURYZBAY          3
Бостандыкский      3
almaly             3
zhetysu            2
Medeuskiy          2
Turksibskiy        2
Турксибский        2
 Nauryzbay         1
Алатауский         1
Ауэзовский         1
Жетысуский         1
alatau             1
Алма

In [9]:
# Показать оригинальные названия, которые стали NaN
mask = df["district"].isna()
print("Количество NaN:", mask.sum())
print("\nОригинальные district для них (до маппинга — но у нас уже замаплено):")
print("Проблема: мы уже перезаписали district. Надо взять из raw.")

# Смотрим на raw данные — какие варианты district не попали ни в один корень
raw_dist = raw["district"].astype(str).str.strip().str.casefold()

def canon_district(s):
    if s.startswith("медеу") or s.startswith("medeu"):  return "Medeu"
    if s.startswith("жетысу") or s.startswith("zhet"):  return "Zhetysu"
    if s.startswith("наурыз") or s.startswith("naur"):  return "Nauryzbay"
    if s.startswith("алмалин") or s.startswith("almal"): return "Almaly"
    if s.startswith("alata"):  return "Alatau"
    if s.startswith("bost"):   return "Bostandyk"
    if s.startswith("auez"):   return "Auezov"
    if s.startswith("turk"):   return "Turksib"
    return None

raw_canon = raw_dist.map(canon_district)
unmatched = raw_dist[raw_canon.isna()].value_counts()
print("\n=== Нераспознанные варианты district ===")
print(unmatched)

Количество NaN: 7

Оригинальные district для них (до маппинга — но у нас уже замаплено):
Проблема: мы уже перезаписали district. Надо взять из raw.

=== Нераспознанные варианты district ===
district
бостандыкский    3
турксибский      3
ауэзовский       2
алатауский       1
Name: count, dtype: int64


In [10]:
# 3b — нормализация district (v2, с русскими)
dist = raw["district"].astype(str).str.strip().str.casefold()

def canon_district(s):
    # Русские названия
    if s.startswith("медеу")   or s.startswith("medeu"):  return "Medeu"
    if s.startswith("жетысу")  or s.startswith("zhet"):   return "Zhetysu"
    if s.startswith("наурыз")  or s.startswith("naur"):   return "Nauryzbay"
    if s.startswith("алмалин") or s.startswith("almal"):  return "Almaly"
    if s.startswith("алатау")  or s.startswith("alata"):  return "Alatau"
    if s.startswith("бостандык") or s.startswith("bost"): return "Bostandyk"
    if s.startswith("ауэзов")  or s.startswith("auez"):   return "Auezov"
    if s.startswith("турксиб") or s.startswith("turk"):   return "Turksib"
    return None

df["district"] = dist.map(canon_district)

print("Уникальных district:", df["district"].nunique())
print("Пропущено (NaN):", df["district"].isna().sum())
print("\nРаспределение:")
print(df["district"].value_counts())

Уникальных district: 8
Пропущено (NaN): 0

Распределение:
district
Medeu        167
Bostandyk    163
Auezov       153
Turksib      150
Zhetysu      141
Almaly       136
Nauryzbay    126
Alatau       123
Name: count, dtype: int64


In [11]:
# 3a — Duplicates and reposts
import pandas as pd

# Считаем "до"
n_before = len(df)
print(f"Строк до очистки: {n_before}")

# 1) Exact duplicates — drop
n_after_exact = len(df.drop_duplicates())
print(f"После drop_duplicates(): {n_after_exact}  (удалено {n_before - n_after_exact})")
df = df.drop_duplicates().reset_index(drop=True)

# 2) Reposts — оставляем самую ПОЗДНЮЮ дату для каждого listing_id
# Сначала парсим дату (пока как строку, для сортировки достаточно формата YYYY-MM-DD)
df["listed_date"] = pd.to_datetime(df["listed_date"])

n_before_repost = len(df)
df = df.sort_values("listed_date").drop_duplicates("listing_id", keep="last").reset_index(drop=True)
n_after_repost = len(df)
print(f"После resolve reposts (keep last date): {n_after_repost}  (удалено {n_before_repost - n_after_repost})")

print(f"\nИтого строк после 3a: {len(df)}")
print("Уникальных listing_id:", df["listing_id"].nunique())
print("Проверка (должно быть 0 дублей listing_id):", df["listing_id"].duplicated().sum())

Строк до очистки: 1159
После drop_duplicates(): 1156  (удалено 3)
После resolve reposts (keep last date): 1080  (удалено 76)

Итого строк после 3a: 1080
Уникальных listing_id: 1080
Проверка (должно быть 0 дублей listing_id): 0


In [12]:
# Проверка: сколько exact duplicates было ДО любых манипуляций
raw_sample = raw[raw["listing_id"].isin(raw["listing_id"].drop_duplicates().sample(frac=0.9, random_state=2534))]
print("Строк в raw_sample:", len(raw_sample))
print("Exact duplicates в raw_sample:", raw_sample.duplicated().sum())

# И сравним с текущим df
print("\nТекущий df:")
print("Строк:", len(df))
print("Exact duplicates:", df.duplicated().sum())

Строк в raw_sample: 1159
Exact duplicates в raw_sample: 33

Текущий df:
Строк: 1080
Exact duplicates: 0


In [13]:
# Проверка состояния после Step 3a и 3b
print("=== Состояние df после 3a и 3b ===")
print(f"Строк: {len(df)}")
print(f"Уникальных listing_id: {df['listing_id'].nunique()}")
print(f"Exact dups: {df.duplicated().sum()}")
print(f"District unique: {df['district'].nunique()}")
print(f"District NaN: {df['district'].isna().sum()}")
print(f"\nDtypes:")
print(df.dtypes)

=== Состояние df после 3a и 3b ===
Строк: 1080
Уникальных listing_id: 1080
Exact dups: 0
District unique: 8
District NaN: 0

Dtypes:
listing_id                  object
listed_date         datetime64[ns]
district                    object
rooms                       object
area_m2                     object
floor                        int64
total_floors                 int64
year_built                 float64
building_type               object
ceiling_height_m           float64
price_kzt                   object
dtype: object


In [14]:
# 3c — Text that should be numbers
print("=== ДО парсинга ===")
for col in ["price_kzt", "area_m2", "rooms"]:
    print(f"{col}: {df[col].astype(str).sample(5, random_state=1).tolist()}")

# price_kzt — оставляем только цифры
df["price_kzt"] = df["price_kzt"].astype(str).str.replace(r"[^\d]", "", regex=True)
df["price_kzt"] = pd.to_numeric(df["price_kzt"], errors="coerce")

# area_m2 — убираем m², NBSP, пробелы; запятая → точка
df["area_m2"] = (df["area_m2"].astype(str)
                 .str.replace("m²", "", regex=False)
                 .str.replace("\u00a0", "", regex=False)
                 .str.replace(" ", "", regex=False)
                 .str.replace(",", ".", regex=False)
                 .str.strip())
df["area_m2"] = pd.to_numeric(df["area_m2"], errors="coerce")

# rooms — "studio" → 1
df["rooms"] = df["rooms"].astype(str).str.strip().str.lower().replace({"studio": "1"})
df["rooms"] = pd.to_numeric(df["rooms"], errors="coerce")

print("\n=== ПОСЛЕ парсинга ===")
print(df[["price_kzt", "area_m2", "rooms"]].dtypes)
print("\nNaN после парсинга:")
print(df[["price_kzt", "area_m2", "rooms"]].isna().sum())
print("\nDescribe:")
print(df[["price_kzt", "area_m2", "rooms"]].describe())

=== ДО парсинга ===
price_kzt: ['39930000', '55670000', '38130000', '30740000', '28,870,000']
area_m2: ['45.4', '45.1', '67.6', '53.3', '43.9']
rooms: ['1', '1', '2', '2', '1']

=== ПОСЛЕ парсинга ===
price_kzt      int64
area_m2      float64
rooms          int64
dtype: object

NaN после парсинга:
price_kzt    0
area_m2      0
rooms        0
dtype: int64

Describe:
          price_kzt      area_m2        rooms
count  1.080000e+03  1080.000000  1080.000000
mean   4.625773e+07    68.931759     2.257407
std    2.499619e+07    60.950878     1.007130
min    1.144000e+07    25.300000     1.000000
25%    2.817000e+07    45.075000     2.000000
50%    4.016500e+07    58.400000     2.000000
75%    5.764000e+07    78.850000     3.000000
max    1.930000e+08   879.000000     5.000000


In [15]:
# 4a — Sentinels: ищем 0, -1, 9999 в числовых колонках
print("=== ПОИСК SENTINEL-ЗНАЧЕНИЙ ===\n")
for col in ["rooms", "area_m2", "floor", "total_floors", "year_built", "ceiling_height_m", "price_kzt"]:
    print(f"--- {col} ---")
    print(f"  == 0:    {(df[col] == 0).sum()}")
    print(f"  == -1:   {(df[col] == -1).sum()}")
    print(f"  == 9999: {(df[col] == 9999).sum()}")
    print(f"  min: {df[col].min()}, max: {df[col].max()}")
    print()

=== ПОИСК SENTINEL-ЗНАЧЕНИЙ ===

--- rooms ---
  == 0:    0
  == -1:   0
  == 9999: 0
  min: 1, max: 5

--- area_m2 ---
  == 0:    0
  == -1:   0
  == 9999: 0
  min: 25.3, max: 879.0

--- floor ---
  == 0:    0
  == -1:   11
  == 9999: 0
  min: -1, max: 30

--- total_floors ---
  == 0:    0
  == -1:   0
  == 9999: 0
  min: 3, max: 25

--- year_built ---
  == 0:    18
  == -1:   0
  == 9999: 0
  min: 0.0, max: 2024.0

--- ceiling_height_m ---
  == 0:    22
  == -1:   0
  == 9999: 0
  min: 0.0, max: 3.2

--- price_kzt ---
  == 0:    0
  == -1:   0
  == 9999: 0
  min: 11440000, max: 193000000



In [16]:
# 4b — Невозможные значения
print("=== НЕВОЗМОЖНЫЕ ЗНАЧЕНИЯ ===\n")
print("floor > total_floors:", (df["floor"] > df["total_floors"]).sum())
print("floor < 1:", (df["floor"] < 1).sum())
print("total_floors < 1:", (df["total_floors"] < 1).sum())
print("year_built > 2026:", (df["year_built"] > 2026).sum())
print("year_built < 1900:", (df["year_built"] < 1900).sum())
print("ceiling_height_m < 2.0:", (df["ceiling_height_m"] < 2.0).sum())
print("ceiling_height_m > 5.0:", (df["ceiling_height_m"] > 5.0).sum())
print("area_m2 < 10:", (df["area_m2"] < 10).sum())
print("area_m2 > 500:", (df["area_m2"] > 500).sum())
print("rooms < 1:", (df["rooms"] < 1).sum())
print("rooms > 10:", (df["rooms"] > 10).sum())

if (df["floor"] > df["total_floors"]).sum() > 0:
    print("\nПримеры floor > total_floors:")
    print(df[df["floor"] > df["total_floors"]][["listing_id","floor","total_floors","building_type"]].head(10))

=== НЕВОЗМОЖНЫЕ ЗНАЧЕНИЯ ===

floor > total_floors: 10
floor < 1: 11
total_floors < 1: 0
year_built > 2026: 0
year_built < 1900: 18
ceiling_height_m < 2.0: 22
ceiling_height_m > 5.0: 0
area_m2 < 10: 0
area_m2 > 500: 9
rooms < 1: 0
rooms > 10: 0

Примеры floor > total_floors:
     listing_id  floor  total_floors building_type
112   ALM-10851     13            12         panel
136   ALM-10944      4             3         brick
296   ALM-10044     30            25      monolith
564   ALM-10420     11             9         panel
700   ALM-10628      6             5         brick
754   ALM-10321     10             5         panel
831   ALM-10266     13            12         panel
888   ALM-10881      8             5         panel
990   ALM-10074     11             9      monolith
1005  ALM-10695     14            12         panel


In [17]:
# 4a (компактно) — sentinels
print("Sentinel-значения (0, -1, 9999):\n")
for col in ["rooms", "area_m2", "floor", "total_floors", "year_built", "ceiling_height_m", "price_kzt"]:
    n0 = (df[col] == 0).sum()
    nm1 = (df[col] == -1).sum()
    n9999 = (df[col] == 9999).sum()
    print(f"{col:20s}  ==0: {n0:3d}   ==-1: {nm1:3d}   ==9999: {n9999:3d}")

Sentinel-значения (0, -1, 9999):

rooms                 ==0:   0   ==-1:   0   ==9999:   0
area_m2               ==0:   0   ==-1:   0   ==9999:   0
floor                 ==0:   0   ==-1:  11   ==9999:   0
total_floors          ==0:   0   ==-1:   0   ==9999:   0
year_built            ==0:  18   ==-1:   0   ==9999:   0
ceiling_height_m      ==0:  22   ==-1:   0   ==9999:   0
price_kzt             ==0:   0   ==-1:   0   ==9999:   0


In [18]:
# ==========================================================
# STEP 4 — Sentinels, impossible values, outliers
# ==========================================================
import numpy as np

log_step4 = {}

# --- 4a: Заменяем sentinels на NaN ---
sentinel_map = {
    "floor": -1,
    "year_built": 0,
    "ceiling_height_m": 0,
}

print("=== 4a · Sentinels → NaN ===")
for col, sentinel in sentinel_map.items():
    n = (df[col] == sentinel).sum()
    df.loc[df[col] == sentinel, col] = np.nan
    log_step4[f"{col} == {sentinel}"] = n
    print(f"  {col} == {sentinel}: {n} строк заменено на NaN")

# --- 4b: Невозможные значения → NaN ---
print("\n=== 4b · Impossible values → NaN ===")

# floor > total_floors (две колонки разом) — обнуляем floor
mask_floor_gt = df["floor"] > df["total_floors"]
print(f"  floor > total_floors: {mask_floor_gt.sum()} строк → floor = NaN")
df.loc[mask_floor_gt, "floor"] = np.nan

# year_built < 1900 — обнуляем (скорее всего опечатка)
mask_year = df["year_built"] < 1900
print(f"  year_built < 1900: {mask_year.sum()} строк → year_built = NaN")
df.loc[mask_year, "year_built"] = np.nan

# ceiling_height_m < 2.0 — обнуляем (нереальный потолок)
mask_ceil = df["ceiling_height_m"] < 2.0
print(f"  ceiling_height_m < 2.0: {mask_ceil.sum()} строк → ceiling_height_m = NaN")
df.loc[mask_ceil, "ceiling_height_m"] = np.nan

# --- 4c: Выбросы по price_per_m2 через IQR (с группировкой по district) ---
print("\n=== 4c · Выбросы (IQR по price_per_m2) ===")

df["area_before_fix"] = df["area_m2"]          # сохраняем для шага 6
df["price_per_m2"] = df["price_kzt"] / df["area_m2"]

def fences(values, groups=None):
    grouped = values if groups is None else values.groupby(groups)
    q1, q3 = grouped.quantile(0.25), grouped.quantile(0.75)
    if groups is not None:
        q1, q3 = groups.map(q1), groups.map(q3)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

# Глобально
low_g, high_g = fences(df["price_per_m2"])
mask_global = (df["price_per_m2"] < low_g) | (df["price_per_m2"] > high_g)
print(f"  Глобально: {mask_global.sum()} выбросов")

# По district
low_d, high_d = fences(df["price_per_m2"], df["district"])
mask_district = (df["price_per_m2"] < low_d) | (df["price_per_m2"] > high_d)
print(f"  По district: {mask_district.sum()} выбросов")

# По rooms+building_type
low_r, high_r = fences(df["price_per_m2"], df["rooms"])
mask_rooms = (df["price_per_m2"] < low_r) | (df["price_per_m2"] > high_r)
print(f"  По rooms: {mask_rooms.sum()} выбросов")

print("\nПримеры выбросов по district:")
print(df[mask_district][["listing_id","district","area_m2","price_kzt","price_per_m2"]].head(10))

print("\n=== Итог после шага 4 ===")
print(f"Строк: {len(df)}")
print("NaN по колонкам:")
print(df[["floor","year_built","ceiling_height_m"]].isna().sum())

=== 4a · Sentinels → NaN ===
  floor == -1: 11 строк заменено на NaN
  year_built == 0: 18 строк заменено на NaN
  ceiling_height_m == 0: 22 строк заменено на NaN

=== 4b · Impossible values → NaN ===
  floor > total_floors: 10 строк → floor = NaN
  year_built < 1900: 0 строк → year_built = NaN
  ceiling_height_m < 2.0: 0 строк → ceiling_height_m = NaN

=== 4c · Выбросы (IQR по price_per_m2) ===
  Глобально: 0 выбросов
  По district: 0 выбросов
  По rooms: 0 выбросов

Примеры выбросов по district:
Empty DataFrame
Columns: [listing_id, district, area_m2, price_kzt, price_per_m2]
Index: []

=== Итог после шага 4 ===
Строк: 1080
NaN по колонкам:
floor                21
year_built           69
ceiling_height_m    152
dtype: int64


In [20]:
# 4c v3 — Выбросы по price_per_m2, группировка по district + rooms
print("=== 4c v3 · Выбросы по price_per_m2 ===\n")

df["price_per_m2"] = df["price_kzt"] / df["area_m2"]

def fences_grouped(values, group_cols):
    """group_cols — список названий колонок в df."""
    q1 = df.groupby(group_cols)["price_per_m2"].quantile(0.25)
    q3 = df.groupby(group_cols)["price_per_m2"].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    # "разворачиваем" результат обратно на каждую строку через merge
    fences_df = pd.DataFrame({"low": low, "high": high}).reset_index()
    merged = df.merge(fences_df, on=group_cols, how="left")
    return merged["low"].values, merged["high"].values

# Глобально
low_g, high_g = fences(df["price_per_m2"])
m_g = (df["price_per_m2"] < low_g) | (df["price_per_m2"] > high_g)
print(f"Глобально: {m_g.sum()} выбросов")

# По district
low_d, high_d = fences(df["price_per_m2"], df["district"])
m_d = (df["price_per_m2"] < low_d) | (df["price_per_m2"] > high_d)
print(f"По district: {m_d.sum()} выбросов")

# По district + rooms
low_dr, high_dr = fences_grouped(df["price_per_m2"], ["district", "rooms"])
m_dr = (df["price_per_m2"] < low_dr) | (df["price_per_m2"] > high_dr)
print(f"По district + rooms: {m_dr.sum()} выбросов  ← самая точная")

# По district + rooms + building_type
low_full, high_full = fences_grouped(df["price_per_m2"], ["district", "rooms", "building_type"])
m_full = (df["price_per_m2"] < low_full) | (df["price_per_m2"] > high_full)
print(f"По district + rooms + building_type: {m_full.sum()} выбросов")

# Примеры выбросов (по district + rooms)
print("\nПримеры выбросов (по district+rooms):")
cols = ["listing_id","district","rooms","area_m2","price_kzt","price_per_m2"]
print(df[m_dr][cols].head(15))

print(f"\nСтатистика выбросов ({m_dr.sum()} шт):")
print(df.loc[m_dr, "price_per_m2"].describe())

print(f"\nСтатистика НЕ выбросов ({len(df) - m_dr.sum()} шт):")
print(df.loc[~m_dr, "price_per_m2"].describe())

=== 4c v3 · Выбросы по price_per_m2 ===

Глобально: 0 выбросов
По district: 0 выбросов
По district + rooms: 3 выбросов  ← самая точная
По district + rooms + building_type: 28 выбросов

Примеры выбросов (по district+rooms):
    listing_id district  rooms  area_m2  price_kzt  price_per_m2
373  ALM-10629   Auezov      3     68.5   86560000  1.263650e+06
650  ALM-10264   Auezov      1    495.0   42680000  8.622222e+04
730  ALM-10783    Medeu      4    120.4  149210000  1.239286e+06

Статистика выбросов (3 шт):
count    3.000000e+00
mean     8.630525e+05
std      6.728651e+05
min      8.622222e+04
25%      6.627540e+05
50%      1.239286e+06
75%      1.251468e+06
max      1.263650e+06
Name: price_per_m2, dtype: float64

Статистика НЕ выбросов (1077 шт):
count    1.077000e+03
mean     7.201664e+05
std      2.471683e+05
min      4.601103e+04
25%      5.199203e+05
50%      6.628070e+05
75%      9.146045e+05
max      1.446293e+06
Name: price_per_m2, dtype: float64


In [21]:
# 4d — Decision: что делаем с выбросами
print("=== 4d · Решение по выбросам ===\n")

# Выбросы по district + rooms (наша лучшая группировка)
low_dr, high_dr = fences_grouped(df["price_per_m2"], ["district", "rooms"])
m_dr = (df["price_per_m2"] < low_dr) | (df["price_per_m2"] > high_dr)

flagged = df[m_dr][["listing_id", "district", "rooms", "area_m2", "price_kzt", "price_per_m2"]]
print(f"Всего выбросов: {len(flagged)}\n")
print(flagged)

# --- РЕШЕНИЕ ---
# 1. ALM-10264 (495 м², 1 комната) — аномалия area: удаляем
bad_area_ids = df[(df["area_m2"] > 400) & (df["rooms"] <= 2)]["listing_id"].tolist()
print(f"\nАномальные area (area > 400 & rooms <= 2): {bad_area_ids}")

n_before = len(df)
df = df[~df["listing_id"].isin(bad_area_ids)].reset_index(drop=True)
n_after = len(df)
print(f"Удалено строк: {n_before - n_after}  ({n_before} → {n_after})")

# 2. Остальные выбросы (373, 730) — оставляем как реальные дорогие объекты
print("\nОстальные 2 выброса оставлены в df — они выглядят как реальные дорогие объекты")

print(f"\n=== Итог после шага 4d ===")
print(f"Строк: {len(df)}")
print("NaN:")
print(df[["floor", "year_built", "ceiling_height_m"]].isna().sum())

=== 4d · Решение по выбросам ===

Всего выбросов: 3

    listing_id district  rooms  area_m2  price_kzt  price_per_m2
373  ALM-10629   Auezov      3     68.5   86560000  1.263650e+06
650  ALM-10264   Auezov      1    495.0   42680000  8.622222e+04
730  ALM-10783    Medeu      4    120.4  149210000  1.239286e+06

Аномальные area (area > 400 & rooms <= 2): ['ALM-10276', 'ALM-11122', 'ALM-10072', 'ALM-10058', 'ALM-10264', 'ALM-10909', 'ALM-10069', 'ALM-10659']
Удалено строк: 8  (1080 → 1072)

Остальные 2 выброса оставлены в df — они выглядят как реальные дорогие объекты

=== Итог после шага 4d ===
Строк: 1072
NaN:
floor                21
year_built           67
ceiling_height_m    152
dtype: int64


### 4d · Decision (Step 4)

**Найдено:**

| Тип | Кол-во | Решение |
|---|---|---|
| Sentinel `floor == -1` | 11 | → NaN |
| Sentinel `year_built == 0` | 18 | → NaN |
| Sentinel `ceiling_height_m == 0` | 22 | → NaN |
| Impossible `floor > total_floors` | 10 | → floor = NaN |
| Выбросы IQR (по district + rooms) | 3 | см. ниже |

**По выбросам:**

- **ALM-10264** (Auezov, 1 комната, 495 м²) — аномально большая площадь при 1 комнате. 
  Удалено ещё 7 строк с `area_m2 > 400 & rooms <= 2` — это ошибки в area_m2 (лишний ноль).
- **ALM-10629** (Auezov, 1.26M ₸/м²) — оставлено: возможно, элитное жильё или особенность объявления.
- **ALM-10783** (Medeu, 1.24M ₸/м²) — оставлено: Medeu — престижный район, реальная цена.

**Итог:** `1080 → 1072` строк. NaN: floor=21, year_built=67, ceiling=152.